# Trie (Prefix Tree) in Python

A Trie is a tree-like data structure for storing strings efficiently. It supports O(L) insertion, search, and prefix lookup (L = string length).

## Topics Covered
- Trie Node Structure
- Insert Operation
- Search Operation
- Prefix Search (startsWith)
- Delete Operation
- Autocomplete
- Word Break Problem
- Trie vs Hash Map for Prefix Problems

---
## 1. Trie Node & Implementation

Each node stores children (dict of character -> node) and a flag indicating if it's the end of a word.

In [ ]:
class TrieNode:
    """Trie node with children and end-of-word marker."""
    def __init__(self):
        self.children = {}  # char -> TrieNode
        self.is_end = False

class Trie:
    """Trie (Prefix Tree) implementation."""
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        """Insert a word into the trie."""
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.is_end = True

    def search(self, word):
        """Return True if word is in the trie."""
        node = self.root
        for ch in word:
            if ch not in node.children:
                return False
            node = node.children[ch]
        return node.is_end

    def startsWith(self, prefix):
        """Return True if any word with given prefix exists."""
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return False
            node = node.children[ch]
        return True

trie = Trie()
trie.insert("apple")
trie.insert("app")
trie.insert("apricot")
print(f"Search 'apple': {trie.search('apple')}")     # True
print(f"Search 'app': {trie.search('app')}")         # True
print(f"Search 'appl': {trie.search('appl')}")       # False
print(f"Starts with 'app': {trie.startsWith('app')}")  # True
print(f"Starts with 'apr': {trie.startsWith('apr')}")  # True
print(f"Starts with 'b': {trie.startsWith('b')}")      # False
# Time: O(L) for all operations, Space: O(N * L)
# Where L = word length, N = number of words

---
## 2. Delete Operation

Delete a word from the trie. Must be careful to clean up unused nodes.

In [ ]:
class TrieWithDelete(Trie):
    """Trie with delete operation."""
    def delete(self, word):
        """Delete a word from trie. Returns True if deleted."""
        def dfs(node, idx):
            if idx == len(word):
                if not node.is_end:
                    return False
                node.is_end = False
                return len(node.children) == 0  # Can be deleted?
            ch = word[idx]
            if ch not in node.children:
                return False
            can_delete = dfs(node.children[ch], idx + 1)
            if can_delete:
                del node.children[ch]
                return len(node.children) == 0 and not node.is_end
            return False
        return dfs(self.root, 0)

td = TrieWithDelete()
td.insert("apple")
td.insert("app")
print(f"After delete 'apple': {td.delete('apple')}")  # True
print(f"Search 'apple': {td.search('apple')}")       # False
print(f"Search 'app': {td.search('app')}")           # True
print(f"Starts with 'app': {td.startsWith('app')}")  # True
# Time: O(L), Space: O(L) for recursion

---
## 3. Autocomplete

Given a prefix, return all words in the trie with that prefix.

In [ ]:
class TrieAutocomplete(Trie):
    """Trie with autocomplete."""
    def autocomplete(self, prefix):
        """Return all words with given prefix."""
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return []
            node = node.children[ch]
        # DFS from node to find all words
        result = []
        def dfs(node, path):
            if node.is_end:
                result.append(prefix + path)
            for ch, child in sorted(node.children.items()):
                dfs(child, path + ch)
        dfs(node, "")
        return result

ta = TrieAutocomplete()
words = ["apple", "app", "application", "appetite", "applaud", "banana"]
for w in words:
    ta.insert(w)
print(f"Autocomplete 'app': {ta.autocomplete('app')}")
print(f"Autocomplete 'appl': {ta.autocomplete('appl')}")
print(f"Autocomplete 'xyz': {ta.autocomplete('xyz')}")
# Time: O(L + K) where K = number of suggestions

---
## 4. Word Break Problem

Given a string and a dictionary, determine if the string can be segmented into dictionary words.

In [ ]:
def word_break(s, word_dict):
    """Check if s can be segmented into words from word_dict."""
    trie = Trie()
    for w in word_dict:
        trie.insert(w)
    n = len(s)
    dp = [False] * (n + 1)
    dp[0] = True
    for i in range(n):
        if dp[i]:
            node = trie.root
            for j in range(i, n):
                if s[j] not in node.children:
                    break
                node = node.children[s[j]]
                if node.is_end:
                    dp[j + 1] = True
    return dp[n]

# Test
print(word_break("leetcode", ["leet", "code"]))  # True
print(word_break("applepenapple", ["apple", "pen"]))  # True
print(word_break("catsandog", ["cats", "dog", "sand", "and", "cat"]))  # False
# Time: O(n²), Space: O(n + total chars in dict)

---
## 5. Trie vs Hash Map for Prefix Problems

| Operation | Trie | Hash Map |
|---|---|---|
| Exact match | O(L) | O(L) to hash, O(1) avg |
| Prefix search | O(L) | O(N * L) iterate all keys |
| Autocomplete | O(L + K) | O(N * L) iterate all keys |
| Memory | More (pointers) | Less |
| Wildcard search | Possible via DFS | Difficult |

In [ ]:
# Wildcard search with trie (e.g., "a..le" or "a.ple")
class TrieWildcard(Trie):
    """Trie that supports '.' wildcard search."""
    def search_wildcard(self, word):
        """Search with '.' matching any character."""
        def dfs(node, idx):
            if idx == len(word):
                return node.is_end
            ch = word[idx]
            if ch == '.':
                for child in node.children.values():
                    if dfs(child, idx + 1):
                        return True
                return False
            if ch not in node.children:
                return False
            return dfs(node.children[ch], idx + 1)
        return dfs(self.root, 0)

tw = TrieWildcard()
tw.insert("apple")
tw.insert("applause")
tw.insert("banana")
print(f"Search '.pple': {tw.search_wildcard('.pple')}")      # True
print(f"Search 'a..le': {tw.search_wildcard('a..le')}")      # True
print(f"Search '....n.': {tw.search_wildcard('....n.')}")    # False

---
## LeetCode Problems

### LeetCode 208 - Implement Trie (Prefix Tree)

**Problem:** Implement a trie with insert, search, and startsWith methods.

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.is_end = True

    def search(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                return False
            node = node.children[ch]
        return node.is_end

    def startsWith(self, prefix):
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return False
            node = node.children[ch]
        return True

# Time: O(L) each, Space: O(N * L)

### LeetCode 211 - Design Add and Search Words Data Structure

**Problem:** Support adding words and searching with '.' wildcard.

In [ ]:
class WordDictionary:
    def __init__(self):
        self.children = {}
        self.is_end = False

    def addWord(self, word):
        node = self
        for ch in word:
            if ch not in node.children:
                node.children[ch] = WordDictionary()
            node = node.children[ch]
        node.is_end = True

    def search(self, word):
        def dfs(node, idx):
            if idx == len(word):
                return node.is_end
            ch = word[idx]
            if ch == '.':
                for child in node.children.values():
                    if dfs(child, idx + 1):
                        return True
                return False
            if ch not in node.children:
                return False
            return dfs(node.children[ch], idx + 1)
        return dfs(self, 0)

wd = WordDictionary()
wd.addWord("bad")
wd.addWord("dad")
wd.addWord("mad")
print(wd.search("pad"))  # False
print(wd.search("bad"))  # True
print(wd.search(".ad"))  # True
print(wd.search("b.."))  # True
# Time: O(L) for add, O(26^L) worst for wildcard search

### LeetCode 212 - Word Search II

**Problem:** Given a board and a list of words, find all words that exist on the board. Use a trie for efficiency.

In [ ]:
def findWords(board, words):
    # Build trie
    trie = {}
    for w in words:
        node = trie
        for ch in w:
            if ch not in node:
                node[ch] = {}
            node = node[ch]
        node['#'] = True  # end marker

    rows, cols = len(board), len(board[0])
    result = set()

    def dfs(r, c, node, path):
        if '#' in node:
            result.add(path)
        if r < 0 or r >= rows or c < 0 or c >= cols:
            return
        ch = board[r][c]
        if ch not in node:
            return
        board[r][c] = '#'  # Mark visited
        for dr, dc in [(1,0), (-1,0), (0,1), (0,-1)]:
            dfs(r + dr, c + dc, node[ch], path + ch)
        board[r][c] = ch

    for r in range(rows):
        for c in range(cols):
            if board[r][c] in trie:
                dfs(r, c, trie, "")
    return list(result)

board = [
    ["o","a","a","n"],
    ["e","t","a","e"],
    ["i","h","k","r"],
    ["i","f","l","v"]
]
words = ["oath","pea","eat","rain"]
print(f"Found words: {findWords(board, words)}")  # ['eat', 'oath']
# Time: O(m * n * 4^L + W * L), Space: O(W * L)

---
## Summary

| Operation | Time | Space | Notes |
|---|---|---|---|
| Insert | O(L) | O(L) | Creates new nodes for each char |
| Search | O(L) | O(1) | Traverse, check is_end |
| StartsWith | O(L) | O(1) | Traverse, no end check |
| Delete | O(L) | O(L) | Recursive cleanup |
| Autocomplete | O(L + K) | O(K) | DFS from prefix node |
| Wildcard | O(26^L) worst | O(L) | DFS branching on '.' |